In [1]:
from pyspark.sql import SparkSession, DataFrame
import pyspark.sql.dataframe
import pyspark.sql.functions as f
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer
import math as m
from pyspark.ml.stat import Correlation
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
spark = (
    SparkSession.builder.appName("Project 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/03 12:11:10 WARN Utils: Your hostname, Lachys-Laptop, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/10/03 12:11:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/03 12:11:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
merchant_transactions=spark.read.parquet('../data/curated/merchant_transactions')

In [8]:
merchant_abn_name=merchant_transactions.groupBy('merchant_abn', 'business').count()
merchant_abn_name

merchant_abn,business,count
93693693468,Ipsum Dolor Corpo...,9
17739089622,Auctor Quis Corp.,17945
95279812400,Eu Dui Cum Ltd,7235
19237425345,A Scelerisque Ass...,12273
10142254217,Arcu Ac Orci Corp...,3036
43660707274,Nulla Integer Vul...,5587
71961434094,Amet Diam Corpora...,29665
80132164373,Integer Sem Corpo...,163
24830778398,Fames Ac Turpis Ltd,692
40252040480,Luctus Felis Puru...,5188


In [9]:
merchant_transactions=merchant_transactions.withColumn("year_month", f.date_format("order_datetime", "yyyy-MM"))
merchant_transactions=merchant_transactions.drop('user_id', 'business', 'order_datetime')

In [4]:
#merchant_transactions

In [14]:
month_agg=(merchant_transactions.groupBy('merchant_abn','year_month', 'biz_tags', 'rev_band')
                                .agg(f.sum('dollar_value').alias('revenue'),
                                     f.mean('take_rate').alias('ave_take_rate')))

In [5]:
#month_agg=month_agg.orderBy('merchant_abn', 'year_month')
#month_agg